# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL, allowing standardized data discovery and access for FAIRness and reproducibility.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# .metadata is an mlcroissant Metadata object; access properties/attributes accordingly
print(f"Dataset Title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"License: {dataset.metadata.license}")

# List available record sets
print("\nAvailable Record Sets (by @id):")
for rs in dataset.metadata.record_sets:
    print(f"- {rs['@id']} : {rs.get('name', '')}")
if not dataset.metadata.record_sets:
    # Some Croissant datasets use 'recordSet' or 'hasPart' directly
    print("(No explicit record sets found in JSON-LD metadata. Fetching from dataset.records() will enumerate record sets.)")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In Croissant, "record sets" are groups of tabular or structured records. Each has fields identified by their `@id`.

In [ ]:
# Enumerate all record sets and their fields
def print_recordset_overview(ds):
    if hasattr(ds.metadata, 'record_sets') and ds.metadata.record_sets:
        for record_set in ds.metadata.record_sets:
            rs_id = record_set['@id']
            fields = record_set.get('fields', [])
            print(f"\nRecord Set @id: {rs_id}")
            print(f"  Name: {record_set.get('name', '(no name)')}")
            print(f"  Description: {record_set.get('description', '(no description)')}")
            print("  Fields:")
            for field in fields:
                if isinstance(field, dict):
                    fid = field.get('@id', 'N/A')
                    label = field.get('name', field.get('description', ''))
                else:
                    fid = field
                    label = ''
                print(f"    - {fid} {label}")
    else:
        print("No explicit record sets found in dataset.metadata.")
        # Print from dataset.records()
        try:
            rs_ids = ds._get_record_set_ids()
            for rs_id in rs_ids:
                print(f"Found Record Set: {rs_id}")
        except Exception as e:
            print("Could not enumerate record sets. Error:", e)

print_recordset_overview(dataset)

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis.

Let's enumerate record sets and select the first one for demonstration.

In [ ]:
# List available record set @ids
if hasattr(dataset.metadata, 'record_sets') and dataset.metadata.record_sets:
    record_set_ids = [rs['@id'] for rs in dataset.metadata.record_sets]
else:
    record_set_ids = dataset._get_record_set_ids()

# For illustration, use the first record set
main_record_set_id = record_set_ids[0]
print(f"Using record set: {main_record_set_id}")

# Load records for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
    else:
        print(f"No records found in record set: {record_set_id}")

if main_record_set_id in dataframes:
    print(f"\nFields in data frame for record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print(f"No DataFrame loaded for record set: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common exploratory transformations, such as filtering records, normalizing numeric fields, and grouping data.

In [ ]:
# We'll try to find a numeric field (e.g., Coverage [%], MW, or peptide count) for demonstration.
df = dataframes[main_record_set_id]
print("Preview of the first few records:")
display(df.head())

# Attempt to find a likely numeric field by type or name
numeric_candidates = [col for col in df.columns if df[col].dtype.kind in 'fi']
if not numeric_candidates:
    # Fallback: find columns with characteristic numeric names
    for possible in ['Coverage [%]', 'coverage', 'MW', 'peptide_count', 'number']:
        if possible in df.columns:
            numeric_candidates.append(possible)

if numeric_candidates:
    numeric_field = numeric_candidates[0]
    print(f"Using numeric field: {numeric_field}")
else:
    raise ValueError("Could not find a numeric field for EDA.")

# Filter records where numeric field > threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (count: {len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to select a grouping (possibly a protein description/type)
possible_group_fields = ['description', 'Protein', 'Gene', 'Class', 'accession', 'accession_number']
group_field = None
for candidate in possible_group_fields:
    if candidate in df.columns:
        group_field = candidate
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable grouping field found for aggregation.")

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field], kde=True, bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If group_field exists, plot average values per group for top N groups
if group_field:
    group_stats = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
    plt.figure(figsize=(10, 6))
    sns.barplot(x=group_stats.values, y=group_stats.index)
    plt.xlabel(f'Average {numeric_field}')
    plt.title(f'Top 10 {group_field} by average {numeric_field}')
    plt.show()

## 6. Conclusion
In this notebook, we loaded the FAIR^2 dataset using the `mlcroissant` library, explored its structure via record sets and fields (referenced by their `@id`s), extracted data into pandas DataFrames, and performed basic filtering, normalization, grouping, and visualization. This approach provides a reproducible and standardized framework for FAIR biomedical data exploration and analysis.

*Notebook generated using `mlcroissant` and dataset metadata from https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json.*